# Installing packages

In [1]:
# install.packages("compositions")
# install.packages("zCompositions")

installiere auch Abhängigkeiten ‘DEoptimR’, ‘tensorA’, ‘robustbase’, ‘bayesm’


Aktualisiere HTML-Index der Pakete in '.Library'

Making 'packages.html' ...
 fertig



In [3]:
packageVersion("compositions")
packageVersion("zCompositions")

[1] ‘2.0.9’

[1] ‘1.6.2’

# CLR

In [1]:
# ==============================================================================
# PROFESSIONAL COMPOSITIONAL DATA ANALYSIS (CLR) PIPELINE - CELL TYPE
# ==============================================================================

options(warn = -1)
options(stringsAsFactors = FALSE)
options(future.globals.maxSize = 10 * 1024^3)

quiet_library <- function(pkg) {
  suppressWarnings(suppressMessages(require(pkg, character.only = TRUE)))
}

pkgs <- c("dplyr", "zCompositions", "compositions", "reshape2", "ggplot2",
          "ggpubr", "tidyr", "tibble", "stringr", "purrr", "rlang")
invisible(lapply(pkgs, quiet_library))

# ==============================================================================
# 1. MASTER CONFIGURATION
# ==============================================================================
work_dir      <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/CLR/v1/cell_type/ref_3month"
metadata_file <- "metadata_for_clr_celltype.csv"
out_prefix    <- "CLR_CellType_Ref_3month"

sample_col    <- "Sample_ID"
target_col    <- "cell_type"
condition_col <- "Condition"

ref_condition       <- "3month-protocol"
min_cells_threshold <- 10    # Keep original cell-type threshold (raw count filter)

condition_order <- c("Rejection", "PCMV/PRV", "Prematurely-terminated",
                     "6month-protocol", "3month-protocol", "Control")

# OPTIONAL: comment out for alphabetical ordering
target_order <- c("CM", "FB", "EC", "Mye_papio", "Lymph_papio", "EC_papio", "MC", "Mye_sus", "Lymph_sus", "NC")

custom_theme <- theme(
  text         = element_text(size = 5, face = "bold", family = "sans", color = "black"),
  axis.text.x  = element_text(angle = 90, hjust = 1, face = "bold", size = 5, family = "sans", color = "black"),
  axis.text.y  = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  axis.title   = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  legend.text  = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  legend.title = element_text(face = "bold", size = 5, family = "sans", color = "black"),
  panel.background = element_blank(),
  panel.grid.major = element_blank(),
  panel.grid.minor = element_blank()
)

our_fill_scale_gradient2 <- scale_fill_gradient2(
  limits = c(-1, 1), oob = scales::squish, high = "#FF6600", mid = "white", low = "#6699FF"
)

setwd(work_dir)
dir.create("results", showWarnings = FALSE)

# ==============================================================================
# 2. DATA PREPARATION
# ==============================================================================
cat("Loading metadata and aggregating counts...\n")
meta <- read.csv(metadata_file, check.names = FALSE)
if (colnames(meta)[1] == "") { meta <- meta[, -1] }

# Ensure no ghost factor levels
meta[[target_col]] <- as.character(meta[[target_col]])
cat("DEBUG - Unique cell types:", paste(unique(meta[[target_col]]), collapse = ", "), "\n")

prop.coarse <- meta %>%
  group_by(!!sym(target_col), !!sym(sample_col), !!sym(condition_col)) %>%
  summarise(count = n(), .groups = "drop")

# --- Raw count matrix (used for min_cells filter only) ---
count_matrix <- reshape2::acast(
  prop.coarse,
  as.formula(paste(sample_col, target_col, sep = "~")),
  value.var = "count", fun.aggregate = mean, fill = 0
)

cat("Debug: cell_type level has", ncol(count_matrix), "types and",
    nrow(count_matrix), "samples.\n")

if (ncol(count_matrix) < 2) stop("CLR requires >= 2 cell types. Check your metadata.")

# --- GBM Bayesian zero imputation --- (preserves compositional geometry)
# This is the statistically correct method when one cell type (CM ~88%)
# heavily dominates the composition. Pseudocount +0.2 would distort CLR
# by shifting the geometric mean and can flip effect size signs for CM/EC/FB.
prop_matrix_raw <- count_matrix
all_missing     <- apply(is.na(prop_matrix_raw), 1, all)
prop_matrix_raw[!all_missing, ] <- as.matrix(
  zCompositions::cmultRepl(prop_matrix_raw[!all_missing, ], method = "GBM")
)
prop_matrix <- prop_matrix_raw  

# Build analysis dataframe
prop_df <- as.data.frame(prop_matrix) %>%
  tibble::rownames_to_column(sample_col) %>%
  left_join(distinct(meta, !!sym(sample_col), !!sym(condition_col)), by = sample_col)
prop_df[[condition_col]] <- trimws(prop_df[[condition_col]])

# ==============================================================================
# 3. DIFFERENTIAL ABUNDANCE STATISTICAL FUNCTION 
# ==============================================================================
da.by.covariate <- function(ct_in, target_name, condition_vec) {
  Z    <- compositions::clr(compositions::acomp(ct_in))
  data <- data.frame(
    abundance  = Z[, target_name],
    proportion = ct_in[, target_name],
    Condition  = condition_vec
  )
  m    <- lm(abundance  ~ Condition, data = data)
  smr  <- t(coef(summary(m))[2, ])
  m2   <- lm(proportion ~ Condition, data = data)
  smr2 <- t(coef(summary(m2))[2, ])
  return(data.frame(smr, smr2,
                    mean.prop      = mean(data$proportion, na.rm = TRUE),
                    mean.prop.ctrl = coef(m2)[1],
                    check.names    = FALSE))
}

# ==============================================================================
# 4. ITERATE CONDITIONS vs REFERENCE
# ==============================================================================
cat("Running CLR and Linear Models...\n")
conditions_to_test <- setdiff(unique(prop_df[[condition_col]]), ref_condition)
all_results <- list()

for (cond in conditions_to_test) {
  selected <- prop_df %>%
    filter(!!sym(condition_col) %in% c(ref_condition, cond)) %>%
    # min_cells filter uses raw count_matrix (pre-imputation) — correct
    filter(!!sym(sample_col) %in% rownames(count_matrix)[rowSums(count_matrix) >= min_cells_threshold])

  if (sum(selected[[condition_col]] == ref_condition) > 0 &&
      sum(selected[[condition_col]] == cond) > 0) {

    selected$Condition_Factor <- factor(selected[[condition_col]], levels = c(ref_condition, cond))
    valid_targets <- setdiff(colnames(selected), c(sample_col, condition_col, "Condition_Factor"))

    results <- lapply(valid_targets, function(t)
      da.by.covariate(selected[, valid_targets, drop = FALSE], t, selected$Condition_Factor))

    res <- do.call(rbind, results)
    rownames(res) <- valid_targets
    all_results[[cond]] <- res
  }
}

if (length(all_results) == 0) stop("No conditions passed filtering. Check metadata and thresholds.")

# Build results_df
results_df <- imap_dfr(all_results, ~ data.frame(
  Condition        = .y,
  Target           = rownames(.x),
  PValue           = .x[, "Pr(>|t|)"],
  EffectSize       = .x[, "Estimate"],
  MeanPropControl  = .x[, "mean.prop.ctrl"],
  stringsAsFactors = FALSE
)) %>%
  group_by(Condition) %>%
  mutate(FDR = p.adjust(PValue, method = "BH")) %>%
  ungroup()

# ==============================================================================
# 5. FOLD CHANGE & LOG2FC
#    fold_change = mean(prop | Condition) / mean(prop | ref_condition)
#    log2FC      = log2(fold_change)
#    Both computed from GBM-imputed prop_matrix — consistent with CLR stats
# ==============================================================================
prop_long_full <- as.data.frame(prop_matrix) %>%
  tibble::rownames_to_column(sample_col) %>%
  tidyr::pivot_longer(-!!sym(sample_col), names_to = "Target", values_to = "Prop") %>%
  dplyr::left_join(dplyr::distinct(meta, !!sym(sample_col), !!sym(condition_col)), by = sample_col)

ref_mean_props <- prop_long_full %>%
  dplyr::filter(!!sym(condition_col) == ref_condition) %>%
  dplyr::group_by(Target) %>%
  dplyr::summarise(mean_prop_ref = mean(Prop, na.rm = TRUE), .groups = "drop")

cond_mean_props <- prop_long_full %>%
  dplyr::filter(!!sym(condition_col) != ref_condition) %>%
  dplyr::group_by(!!sym(condition_col), Target) %>%
  dplyr::summarise(mean_prop_cond = mean(Prop, na.rm = TRUE), .groups = "drop") %>%
  dplyr::rename(Condition = !!sym(condition_col))

fc_df <- cond_mean_props %>%
  dplyr::left_join(ref_mean_props, by = "Target") %>%
  dplyr::mutate(
    fold_change = mean_prop_cond / mean_prop_ref,
    log2FC      = log2(fold_change)
  ) %>%
  dplyr::select(Condition, Target, fold_change, log2FC)

results_df <- results_df %>%
  dplyr::left_join(fc_df, by = c("Condition", "Target"))

# ==============================================================================
# 6. SAVE PER-ANALYSIS CSV  +  MASTER CSV
# ==============================================================================
write.csv(results_df,
          file.path("results", paste0(out_prefix, "_Stats.csv")),
          row.names = FALSE)
cat("Statistics saved to:", file.path("results", paste0(out_prefix, "_Stats.csv")), "\n")

# Master CSV with renamed columns + Ref_condition
master_df <- results_df %>%
  dplyr::rename(cell_type = Target) %>%
  dplyr::mutate(Ref_condition = ref_condition) %>%
  dplyr::select(
    Condition, Ref_condition, cell_type,
    PValue, FDR, EffectSize, MeanPropControl,
    fold_change, log2FC
  )

master_out <- file.path("results", paste0(out_prefix, "_WholeMaster.csv"))
write.csv(master_df, master_out, row.names = FALSE)
cat("Master CSV saved to:", master_out, "\n")

# ==============================================================================
# 7. VISUALIZATION - HEATMAP
# ==============================================================================
cat("Generating visuals...\n")

results_df$Condition <- factor(results_df$Condition, levels = condition_order)
if (exists("target_order") && !is.null(target_order)) {
  results_df$Target <- factor(results_df$Target, levels = target_order)
}

p1 <- ggplot(results_df, aes(x = Target, y = Condition, fill = EffectSize)) +
  geom_tile(color = "white", linewidth = 0.15) +
  our_fill_scale_gradient2 +
  labs(y = "Condition", x = target_col,
       fill = "Proportion change\nrelative to Reference") +
  custom_theme + theme(legend.position = "bottom") +
  geom_text(
    data = subset(results_df, FDR < 0.05),
    aes(label = gsub("e-0", "e-", formatC(PValue, format = "e", digits = 1))),
    color = "black", size = 5 / .pt, fontface = "bold", family = "sans"
  )

ggsave(file.path("results", paste0(out_prefix, "_Heatmap.pdf")),
       plot = p1, width = 3.5, height = 2.5, units = "in", dpi = 400)

# ==============================================================================
# 8. VISUALIZATION - REFERENCE BARPLOT
# ==============================================================================
prop_long <- as.data.frame(prop_matrix) %>%
  tibble::rownames_to_column(sample_col) %>%
  tidyr::pivot_longer(-!!sym(sample_col), names_to = "Target", values_to = "Prop")

ref_means <- prop_long %>%
  dplyr::left_join(dplyr::distinct(meta, !!sym(sample_col), !!sym(condition_col)), by = sample_col) %>%
  dplyr::filter(!!sym(condition_col) == ref_condition) %>%
  dplyr::group_by(Target) %>%
  dplyr::summarise(MeanProp = mean(Prop, na.rm = TRUE), .groups = "drop")

if (exists("target_order") && !is.null(target_order)) {
  ref_means$Target <- factor(ref_means$Target, levels = target_order)
}

p_ref <- ggplot(ref_means, aes(x = Target, y = MeanProp, fill = Target)) +
  geom_col(color = "black", alpha = 0.8) +
  geom_text(aes(label = scales::percent(MeanProp, accuracy = 0.1)),
            vjust = -0.5, size = 5 / .pt, fontface = "bold", family = "sans", color = "black") +
  labs(y = paste("Mean % in", ref_condition), x = target_col) +
  theme_bw() + custom_theme +
  theme(axis.title.x = element_blank(), legend.position = "none") +
  scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                     expand = expansion(mult = c(0, 0.15)))

ggsave(file.path("results", paste0(out_prefix, "_Barplot.pdf")),
       plot = p_ref, width = 3.5, height = 2.5, units = "in", dpi = 400)

p_ref_no_x <- p_ref +
  theme(axis.title.x = element_blank(),
        axis.text.x  = element_blank(),
        axis.ticks.x = element_blank())

# ==============================================================================
# 9. VISUALIZATION - COMBINED PLOT
# ==============================================================================
plot_all <- ggarrange(
  p_ref_no_x, p1,
  ncol = 1, nrow = 2,
  align = "v",
  heights = c(2.25, 3)
)

ggsave(file.path("results", paste0(out_prefix, "_Combined_Master.pdf")),
       plot = plot_all, width = 3.5, height = 4, units = "in", dpi = 400)

cat("Pipeline complete. All files saved to:", file.path(work_dir, "results"), "\n")

Loading metadata and aggregating counts...
DEBUG - Unique cell types: CM, NC, MC, FB, EC, Mye_sus, Lymph_sus, Lymph_papio, Mye_papio, EC_papio 
Debug: cell_type level has 10 types and 26 samples.


No. adjusted imputations:  8 




Running CLR and Linear Models...
Statistics saved to: results/CLR_CellType_Ref_3month_Stats.csv 
Master CSV saved to: results/CLR_CellType_Ref_3month_WholeMaster.csv 
Generating visuals...
Pipeline complete. All files saved to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/CLR/v1/cell_type/ref_3month/results 
